# Data_yield.ipynb

ダウンロード禁止によりローカルでの作業ができないですが、どうせ世の中にあるデータです  
そのため、世の中に公開されているデータから同じデータを作ってしまえばセーフでしょう！という試みです  
これを用いて作成したデータはダウンロードしてもOKなデータになります

In [ ]:
# ライブラリインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# # お手本データの読み込み
# 作成したデータと比較するためのお手本データを読み込みます
# 確認したい場合は以下のコメントアウトを外してください
# # パスは適宜変更して下さい
# DATA_PATH = "path/to/your/speciman.csv"  # お手本データのCSVファイルのパス
# df_speciman = pd.read_csv(DATA_PATH)
# df_speciman["Unnamed: 0"] = pd.to_datetime(df_speciman["Unnamed: 0"])
# df_speciman.rename(columns={"Unnamed: 0": "Date"}, inplace=True)
# print(df_speciman.head(-1))

# print(df_speciman.columns)

In [38]:
#　公開データの取得、加工
df_sp500 = yf.download("^GSPC", start="1965-12-25", end="2026-05-15")
df_sp500.columns = [col[0] if isinstance(col, tuple) else col for col in df_sp500.columns]
df_sp500 = df_sp500.reset_index()
df_sp500["Date"] = pd.to_datetime(df_sp500["Date"])
df_sp500 = df_sp500[["Date", "Close"]].rename(columns={"Close": "sp500_abs"})

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10"
df_dgs10 = pd.read_csv(url)
df_dgs10["observation_date"] = pd.to_datetime(df_dgs10["observation_date"])
df_dgs10 = df_dgs10.rename(columns={"DGS10": "DGS10_abs"})
df_dgs10["DGS10_abs"] = pd.to_numeric(df_dgs10["DGS10_abs"], errors="coerce")

df = pd.merge(df_sp500, df_dgs10, left_on="Date", right_on="observation_date", how="inner")
df = df[["Date", "sp500_abs", "DGS10_abs"]].set_index("Date")

df["sp500"] = df["sp500_abs"].pct_change()
df["DGS10"] = df["DGS10_abs"].diff()

START_DATE = "1966-01-02"
df = df[df.index >= START_DATE].dropna()

df = df.reset_index()


df = df[df["Date"].isin(df_speciman["Date"])]  
print(df.head(-1))
print(df.columns)
print(f"行数: {len(df)}")

In [ ]:
# 比較確認用コード
# 確認したい場合は一個目のセルと合わせてコメントアウトを解除して下さい

# df_mine   = df         
# df_sample = df_speciman  

# compare_columns = {
#     "sp500_mine": "sp500_sample",  # S&P500の価格同士
#     "DGS10_mine": "DGS10_sample"   # 10年債利回り同士

# }

# # ==========================================
# # 自動検証処理
# # ==========================================
# print("🔍 データの突合チェックを開始します...\n")

# df_check = pd.merge(
#     df_mine, 
#     df_sample, 
#     on="Date", 
#     how="inner", 
#     suffixes=("_mine", "_sample") 
# )

# for col_mine, col_sample in compare_columns.items():
#     print(f"--- 【{col_mine}】 vs 【{col_sample}】 のチェック ---")
   
#     valid_rows = df_check[df_check[col_mine].notna() & df_check[col_sample].notna()]
    
#     # 引き算して差分を出す（完全に一致していれば 0 になる）
#     diff = (valid_rows[col_mine] - valid_rows[col_sample]).abs()
#     error_rows = valid_rows[diff > 1e-5]
    
#     if len(error_rows) == 0:
#         print(f"✅ 完璧に一致！ (共通データ {len(valid_rows)} 件すべてでズレはありません)")
#     else:
#         print(f"❌ 警告: {len(error_rows)} 件のデータにズレが見つかりました。")
#         print("↓ ズレが発生している主な日付と値 ↓")
#         print(error_rows[["Date", col_mine, col_sample]].head())
#     print("\n" + "="*50 + "\n")

In [ ]:
df.to_csv("path/to/your/output.csv", index=False)